# Sinhala Voice Agent — Colab demo

Mic → VAD → Whisper (Sinhala) → LLM → chunker → SinhalaVITS → speaker.

**Before you start:** set `Runtime → Change runtime type → T4 GPU`, and add your
`GEMINI_API_KEY` (and optionally `HF_TOKEN`) under the 🔑 **Secrets** tab in the left sidebar,
with *Notebook access* enabled.

Run the cells in order. Colab resets on disconnect, so models re-download each session.

## 1. Check the runtime

In [ ]:
!nvidia-smi || echo 'No GPU — set Runtime > Change runtime type > T4 GPU'
import sys; print('Python', sys.version)

## 2. Get the code

Clones <https://github.com/Nethmini-Rathnayake/sinhala-voice-agent>. Edit `REPO_URL` if you
work from a fork, or skip this cell if you uploaded the folder to `/content/sinhala-voice-agent`.


In [ ]:
REPO_URL = 'https://github.com/Nethmini-Rathnayake/sinhala-voice-agent.git'
PROJECT_DIR = '/content/sinhala-voice-agent'

import os, subprocess
if not os.path.isdir(PROJECT_DIR):
    if not REPO_URL:
        raise SystemExit('Set REPO_URL above, or upload the project to ' + PROJECT_DIR)
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
print('working in', os.getcwd())
!ls

## 3. Install dependencies

Two commands, not one: `silero-vad` declares `torchaudio<2.10`, which would drag torch back
and replace Colab's CUDA build. We don't use the part of it that needs torchaudio, so it is
installed without its dependencies.

This takes a few minutes and pip will warn about conflicting versions — that is expected.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-deps silero-vad==6.2.2
print('done — if pip asks to restart the session, do it, then re-run cells 2 and 4 onwards')

## 4. Model cache and API keys

### Adding secrets in Colab

1. Click the **🔑 key icon** in the narrow icon strip on the far left edge of the page
   (below ☰ contents, 🔍 search, {x} variables).
2. Click **+ Add new secret**.
3. **Name:** `GEMINI_API_KEY` — spelled exactly like that. **Value:** paste your key.
4. Switch on **Notebook access** for this notebook, or the cell below cannot read it.

Get a key from <https://aistudio.google.com/apikey> → **Create API key**. Create a *new* one:
since September 2026 the Gemini API rejects older 'standard' keys.

Optional secrets: `HF_TOKEN` (faster, less rate-limited model downloads) and `GROQ_API_KEY`
(alternative provider, with `LLM_PROVIDER=groq`).

**No key icon?** Run the cell anyway — it falls back to a hidden prompt. Keys typed there are
never written into the notebook file.

The cell also points `HF_HOME` at `/content/hf_cache`, so every model download lands on the
Colab disk in one place. Set it *before* importing anything that touches Hugging Face.

In [ ]:
import os
from getpass import getpass

# Model cache: set before importing transformers / huggingface_hub anywhere.
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)


def load_secret(name, required=False):
    """Colab Secrets first, then the existing environment, then a hidden prompt."""
    value = os.environ.get(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None          # no Secrets tab, or notebook access not granted
    if not value and required:
        value = getpass(f'{name} (input hidden): ').strip()
    if value:
        os.environ[name] = value
    print(f'{name}: {"set" if value else "not set"}')
    return value


os.environ.setdefault('LLM_PROVIDER', 'gemini')
provider = os.environ['LLM_PROVIDER']
load_secret('GEMINI_API_KEY', required=provider == 'gemini')
load_secret('GROQ_API_KEY', required=provider == 'groq')
load_secret('HF_TOKEN')       # optional

# If model downloads stall at 0 B, uncomment and re-run cell 5:
# os.environ['HF_HUB_DISABLE_XET'] = '1'
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GROQ_API_KEY'), 'No LLM API key'
print('HF_HOME:', os.environ['HF_HOME'])

## 5. Warm up the models

Downloads ~1 GB for Whisper and ~1 GB for SinhalaVITS on the first run, then loads all three
models and pushes one dummy sentence through TTS and STT. That first pass is always the slow
one (CUDA kernels, lazy weights), so paying it here keeps the first real turn fast.

It doubles as a stage test: the dummy audio goes TTS → VAD → STT, so if the transcript comes
back resembling the input sentence, all three stages work.

In [ ]:
import logging, time
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
for noisy in ('httpx', 'httpx2', 'TTS', 'urllib3'):
    logging.getLogger(noisy).setLevel(logging.WARNING)

from src import stt, tts, vad
from src.config import select_device
print('device:', select_device())

t0 = time.perf_counter(); vad.load();   print(f'VAD   loaded in {time.perf_counter()-t0:6.1f} s')
t0 = time.perf_counter(); stt.get_asr(); print(f'STT   loaded in {time.perf_counter()-t0:6.1f} s ({stt.loaded_model_id()})')
t0 = time.perf_counter(); tts.load();   print(f'TTS   loaded in {time.perf_counter()-t0:6.1f} s')

### Dummy turn through TTS → VAD → STT

In [ ]:
from IPython.display import Audio, display

sentence = 'ආයුබෝවන්, මම ඔබට උදව් කරන්නම්.'

t0 = time.perf_counter(); sr, wav = tts.synthesize(sentence)
print(f'TTS warm-up: {time.perf_counter()-t0:5.1f} s for {len(wav)/sr:.2f} s of audio')
print('romanized:', tts.romanize(sentence))
display(Audio(wav, rate=sr))

speech = vad.trim(wav, sr)
print('after VAD:', f'{len(speech)/16000:.2f} s' if speech is not None else 'no speech found')

t0 = time.perf_counter()
heard = stt.transcribe(speech if speech is not None else wav, 16000 if speech is not None else sr)
print(f'STT warm-up: {time.perf_counter()-t0:5.1f} s')
print('said: ', sentence)
print('heard:', heard)
print('\nModels are warm — the first real turn will be much faster.')

## 6. Test the LLM on its own

Checks streaming and the Sinhala-only rule. A warning here means Latin characters slipped
through into the reply.

In [ ]:
from src import llm
import time

start = time.perf_counter(); first = None
for token in llm.stream_reply([], 'ඔයාට කොහොමද?'):
    if first is None:
        first = time.perf_counter() - start
    print(token, end='')
print(f'\n\n[first token {first*1000:.0f} ms, total {(time.perf_counter()-start)*1000:.0f} ms]')

## 7. One full turn

Record yourself speaking Sinhala, or upload a WAV. The recorder needs microphone permission.

In [ ]:
from google.colab import files

uploaded = files.upload()   # pick a .wav file
input_path = next(iter(uploaded))
print('using', input_path)

In [ ]:
import soundfile as sf
from IPython.display import Audio, display
from src.pipeline import run_turn, reply_sample_rate

audio, file_sr = sf.read(input_path, dtype='float32')
history = []
reply_audio, transcript, reply_text, timings = run_turn(audio, file_sr, history)

print('user: ', transcript)
print('agent:', reply_text)
for key, value in timings.items():
    print(f'  {key:<26} {value:8.0f} ms')
if len(reply_audio):
    display(Audio(reply_audio, rate=reply_sample_rate()))

## 8. Launch the app

`COLAB=1` makes the app request a public share link, printed below as a `*.gradio.live` URL
that works outside Colab for about 72 hours. The app keeps running while this cell runs;
stop the cell to shut it down.

Allow microphone access when the browser asks.

In [ ]:
import os
os.environ['COLAB'] = '1'

from src.app import main
main()

## 9. Latency log and evaluation

In [ ]:
!python -m src.timing logs/latency.jsonl || echo 'no turns logged yet'
# Batch-run every WAV in eval/utterances/ (eval/run_eval.py is not implemented yet):
# !python -m eval.run_eval

## Troubleshooting

| Symptom | Fix |
|---|---|
| Downloads hang at 0 B | Set `HF_HUB_DISABLE_XET=1` in cell 4 and re-run |
| No 🔑 Secrets tab in the sidebar | Cell 4 prompts for the key instead; nothing else to do |
| Slow downloads / rate limits | Add `HF_TOKEN` to the Secrets tab |
| `ModuleNotFoundError` after installing | Colab asked to restart the session; restart, then re-run from cell 2 |
| `OSError: Could not load libtorchcodec` | FFmpeg is missing (`!apt-get install -y ffmpeg`); it is normally present in Colab |
| STT falls back to another model | The primary model's download failed; re-run cell 5 |
| Transcript repeats one character | Whisper repetition loop; lower `stt.max_new_tokens_per_s` in `config.yaml` |
| Reply contains English letters | Check the warning from `src.llm`; the TTS cannot pronounce them |
